In [ ]:
"/experiment2_paraphrasing_results.xlsx"

'/experiment2_paraphrasing_results.xlsx'

In [ ]:
import pandas as pd

df = pd.read_excel("/experiment2_paraphrasing_results.xlsx")

print("Rows:", len(df))
df.head()

Rows: 379


,id,prompt,before_z_score,after_z_score,z_score_drop,before_prediction,after_prediction,original_text,paraphrased_text
0,0,Explain artificial intelligence in simple terms.,10.248630,4.811252,5.437377,True,True,Explain artificial intelligence in simple term...,Arrange for example that one person can solve ...
1,1,Describe why artificial intelligence is import...,12.289081,3.790490,8.498591,True,False,Describe why artificial intelligence is import...,So we can assume this about we are not yet rea...
2,2,Give a beginner-friendly overview of artificia...,11.381159,6.041281,5.339878,True,True,Give a beginner-friendly overview of artificia...,A beginner-friendly understanding of AI will h...
3,3,Explain the main benefits and risks of artific...,0.000000,0.816497,-0.816497,False,False,Explain the main benefits and risks of artific...,Explain the main benefits and risks of artific...
4,4,Describe how artificial intelligence affects e...,13.867505,5.483160,8.384345,True,True,Describe how artificial intelligence affects e...,If you'd like to learn more about artificial i...


In [ ]:
!pip install transformers sentencepiece accelerate -q

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

print(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

cpu


In [ ]:
def extreme_rewrite(text):

    prompt = f"""
Rewrite the following text while preserving exactly the same meaning.

Rules:
- Completely change sentence structure.
- Replace words with different expressions.
- Merge and split sentences.
- Use different grammar.
- Avoid copying phrases whenever possible.
- Keep the meaning unchanged.

Text:
{text}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = model.generate(
        **inputs,
        do_sample=True,
        temperature=1.5,
        top_p=0.95,
        top_k=100,
        max_new_tokens=256
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
sample = df.loc[0, "original_text"]

print("ORIGINAL\n")
print(sample)

print("\n=============================\n")

rewrite = extreme_rewrite(sample)

print("EXTREME REWRITE\n")
print(rewrite)

ORIGINAL

Explain artificial intelligence in simple terms.

To understand how this will work, let's assume a person is a person who can solve a problem and can understand it better. One person might understand the problem better than the other and would be able to understand it better, and there might be a problem in the relationship that will lead to that.

Consider that person and the relationship could be different. Would that person be able to understand that person better and understand it better in the relationship? Would the person be able to understand that relationship better in the relationship? For example, might a person understand better that he was in the relationship and understand better with that person than with the person who understands that relationship better? Would the person in the relationship be able to understand that relationship better in


EXTREME REWRITE

Let's just have somebody sees his personal future. That's what artificial intelligence is going to he

In [ ]:
def extreme_rewrite_v2(text):
    prompt = f"""
You are rewriting text to preserve meaning but remove stylistic and wording traces.

Rewrite the text completely.

Rules:
- Do not copy sentence structure.
- Do not copy phrases longer than 3 words.
- Change word choice aggressively.
- Reorder ideas if meaning remains the same.
- Compress repetitive parts.
- Use a different explanatory style.
- Keep factual meaning unchanged.
- Output only the rewritten text.

Text:
{text}
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = model.generate(
        **inputs,
        do_sample=True,
        temperature=1.7,
        top_p=0.98,
        top_k=120,
        max_new_tokens=256
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
sample = df.loc[0, "original_text"]

rewrite = extreme_rewrite_v2(sample)

print("ORIGINAL:")
print(sample[:1000])

print("\nEXTREME REWRITE:")
print(rewrite)

ORIGINAL:
Explain artificial intelligence in simple terms.

To understand how this will work, let's assume a person is a person who can solve a problem and can understand it better. One person might understand the problem better than the other and would be able to understand it better, and there might be a problem in the relationship that will lead to that.

Consider that person and the relationship could be different. Would that person be able to understand that person better and understand it better in the relationship? Would the person be able to understand that relationship better in the relationship? For example, might a person understand better that he was in the relationship and understand better with that person than with the person who understands that relationship better? Would the person in the relationship be able to understand that relationship better in

EXTREME REWRITE:
Try artificial intelligence to explain what to you will about it. Understand how smart people think yo

In [ ]:
original_detection = detector.detect(sample)
extreme_detection = detector.detect(rewrite)

print("Before z-score:", original_detection["z_score"])
print("Before prediction:", original_detection["prediction"])

print("Extreme rewrite z-score:", extreme_detection["z_score"])
print("Extreme rewrite prediction:", extreme_detection["prediction"])

print("Drop:", original_detection["z_score"] - extreme_detection["z_score"])

NameError: name 'detector' is not defined

In [ ]:
original_detection = detector.detect(sample)
extreme_detection = detector.detect(rewrite)

print("Before z-score:", original_detection["z_score"])
print("Before prediction:", original_detection["prediction"])

print("Extreme rewrite z-score:", extreme_detection["z_score"])
print("Extreme rewrite prediction:", extreme_detection["prediction"])

print("Drop:", original_detection["z_score"] - extreme_detection["z_score"])

NameError: name 'detector' is not defined

In [ ]:
!git clone https://github.com/jwkirchenbauer/lm-watermarking
%cd lm-watermarking
!pip install -r requirements.txt -q

Cloning into 'lm-watermarking'...
remote: Enumerating objects: 351, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 351 (delta 32), reused 16 (delta 16), pack-reused 294 (from 2)
Receiving objects: 100% (351/351), 12.01 MiB | 18.83 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/lm-watermarking


In [ ]:
from transformers import AutoTokenizer
from extended_watermark_processor import WatermarkDetector

In [ ]:
detector_tokenizer = AutoTokenizer.from_pretrained("gpt2")

vocab = list(detector_tokenizer.get_vocab().values())

gamma = 0.25

detector = WatermarkDetector(
    vocab=vocab,
    gamma=gamma,
    seeding_scheme="selfhash",
    device=device,
    tokenizer=detector_tokenizer,
    z_threshold=4.0,
    normalizers=[],
    ignore_repeated_ngrams=False,
    select_green_tokens=True
)

print("Detector ready!")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Detector ready!


In [ ]:
original_detection = detector.detect(sample)
extreme_detection = detector.detect(rewrite)

print("Before z-score:", original_detection["z_score"])
print("Before prediction:", original_detection["prediction"])

print("Extreme rewrite z-score:", extreme_detection["z_score"])
print("Extreme rewrite prediction:", extreme_detection["prediction"])

print("Drop:", original_detection["z_score"] - extreme_detection["z_score"])

Before z-score: 10.24862959629972
Before prediction: True
Extreme rewrite z-score: 1.990614683076411
Extreme rewrite prediction: False
Drop: 8.25801491322331


In [ ]:
extreme_results = []

for i in range(len(df)):
    if i % 10 == 0:
        print(f"Running {i+1}/{len(df)}")

    original = df.loc[i, "original_text"]
    before = detector.detect(original)

    rewritten = extreme_rewrite_v2(original)
    after = detector.detect(rewritten)

    extreme_results.append({
        "id": df.loc[i, "id"],
        "prompt": df.loc[i, "prompt"],
        "before_z_score": before["z_score"],
        "extreme_z_score": after["z_score"],
        "z_score_drop": before["z_score"] - after["z_score"],
        "before_prediction": before["prediction"],
        "extreme_prediction": after["prediction"],
        "original_text": original,
        "extreme_rewrite_text": rewritten
    })

extreme_df = pd.DataFrame(extreme_results)

print("Before detection rate:", extreme_df["before_prediction"].mean())
print("Extreme detection rate:", extreme_df["extreme_prediction"].mean())
print("Average before z-score:", extreme_df["before_z_score"].mean())
print("Average extreme z-score:", extreme_df["extreme_z_score"].mean())
print("Average z-score drop:", extreme_df["z_score_drop"].mean())

extreme_df.to_excel("experiment3_extreme_rewrite_results.xlsx", index=False)

Running 1/379
Running 11/379
Running 21/379


ValueError: Must have at least 1 token to score after the first min_prefix_len=4 tokens required by the seeding scheme.

In [ ]:
extreme_results = []

for i in range(len(df)):
    if i % 10 == 0:
        print(f"Running {i+1}/{len(df)}")

    original = df.loc[i, "original_text"]
    before = detector.detect(original)

    try:
        rewritten = extreme_rewrite_v2(original)

        if rewritten is None or len(rewritten.split()) < 20:
            after_z = None
            after_pred = None
            error = "too_short_or_empty"
        else:
            after = detector.detect(rewritten)
            after_z = after["z_score"]
            after_pred = after["prediction"]
            error = None

    except Exception as e:
        rewritten = ""
        after_z = None
        after_pred = None
        error = str(e)

    extreme_results.append({
        "id": df.loc[i, "id"],
        "prompt": df.loc[i, "prompt"],
        "before_z_score": before["z_score"],
        "extreme_z_score": after_z,
        "z_score_drop": before["z_score"] - after_z if after_z is not None else None,
        "before_prediction": before["prediction"],
        "extreme_prediction": after_pred,
        "error": error,
        "original_text": original,
        "extreme_rewrite_text": rewritten
    })

    pd.DataFrame(extreme_results).to_excel(
        "experiment3_extreme_rewrite_checkpoint.xlsx",
        index=False
    )

extreme_df = pd.DataFrame(extreme_results)

valid = extreme_df.dropna(subset=["extreme_z_score"])

print("Total rows:", len(extreme_df))
print("Valid rows:", len(valid))
print("Before detection rate:", valid["before_prediction"].mean())
print("Extreme detection rate:", valid["extreme_prediction"].mean())
print("Average before z-score:", valid["before_z_score"].mean())
print("Average extreme z-score:", valid["extreme_z_score"].mean())
print("Average z-score drop:", valid["z_score_drop"].mean())

extreme_df.to_excel("experiment3_extreme_rewrite_results.xlsx", index=False)

Running 1/379
Running 11/379
Running 21/379
Running 31/379
Running 41/379
Running 51/379
Running 61/379
Running 71/379
Running 81/379
Running 91/379
Running 101/379
Running 111/379
Running 121/379


In [1]:
import pandas as pd

checkpoint = pd.read_excel("experiment3_extreme_rewrite_checkpoint.xlsx")

print("Saved rows:", len(checkpoint))
checkpoint.tail()

FileNotFoundError: [Errno 2] No such file or directory: 'experiment3_extreme_rewrite_checkpoint.xlsx'

In [2]:
import os

for root, dirs, files in os.walk("/content"):
    for f in files:
        if "experiment3" in f and f.endswith(".xlsx"):
            print(os.path.join(root, f))


/content/lm-watermarking/experiment3_extreme_rewrite_checkpoint.xlsx


In [3]:
import pandas as pd

checkpoint = pd.read_excel("/content/lm-watermarking/experiment3_extreme_rewrite_checkpoint.xlsx")

print("Saved rows:", len(checkpoint))
checkpoint.tail()


Saved rows: 256


,id,prompt,before_z_score,extreme_z_score,z_score_drop,before_prediction,extreme_prediction,error,original_text,extreme_rewrite_text
251,251,Describe why water pollution is important.,17.111965,2.435123,14.676842,True,0.0,NaN,Describe why water pollution is important.\n\n...,If and when a river that is had to be polluted...
252,252,Give a beginner-friendly overview of water pol...,10.090987,-0.742307,10.833295,True,0.0,NaN,Give a beginner-friendly overview of water pol...,water comes from everywhere and was not usuall...
253,253,Explain the main benefits and risks of water p...,5.339795,2.038099,3.301696,True,0.0,NaN,Explain the main benefits and risks of water p...,The data included in the article discussed jus...
254,254,Describe how water pollution affects everyday ...,9.984604,NaN,NaN,True,NaN,too_short_or_empty,Describe how water pollution affects everyday ...,Water Pollution for Female Mamatologists the w...
255,255,Explain the future of water pollution.,10.805116,0.000000,10.805116,True,0.0,NaN,Explain the future of water pollution.\n\nWe a...,"In regard to water, water flooding is actually..."


In [4]:
extreme_results = checkpoint.to_dict("records")

start_idx = len(extreme_results)

print("Resuming from row:", start_idx)

Resuming from row: 256


In [5]:
for i in range(start_idx, len(df)):

    if i % 10 == 0:
        print(f"Running {i+1}/{len(df)}")

    original = df.loc[i, "original_text"]
    before = detector.detect(original)

    try:
        rewritten = extreme_rewrite_v2(original)

        if rewritten is None or len(rewritten.split()) < 20:
            after_z = None
            after_pred = None
            error = "too_short_or_empty"
        else:
            after = detector.detect(rewritten)
            after_z = after["z_score"]
            after_pred = after["prediction"]
            error = None

    except Exception as e:
        rewritten = ""
        after_z = None
        after_pred = None
        error = str(e)

    extreme_results.append({
        "id": df.loc[i, "id"],
        "prompt": df.loc[i, "prompt"],
        "before_z_score": before["z_score"],
        "extreme_z_score": after_z,
        "z_score_drop": before["z_score"] - after_z if after_z is not None else None,
        "before_prediction": before["prediction"],
        "extreme_prediction": after_pred,
        "error": error,
        "original_text": original,
        "extreme_rewrite_text": rewritten
    })

    pd.DataFrame(extreme_results).to_excel(
        "/content/lm-watermarking/experiment3_extreme_rewrite_checkpoint.xlsx",
        index=False
    )

NameError: name 'df' is not defined

In [6]:
%whos

Variable          Type         Data/Info
----------------------------------------
checkpoint        DataFrame          id                 <...>\n[256 rows x 10 columns]
dirs              list         n=0
extreme_results   list         n=256
f                 str          california_housing_test.csv
files             list         n=6
os                module       <module 'os' (frozen)>
pd                module       <module 'pandas' from '/u<...>ages/pandas/__init__.py'>
root              str          /content/sample_data
start_idx         int          256


In [7]:
import pandas as pd

df = pd.read_excel("/experiment2_paraphrasing_results.xlsx")

print(len(df))

379


In [8]:
%whos

Variable          Type         Data/Info
----------------------------------------
checkpoint        DataFrame          id                 <...>\n[256 rows x 10 columns]
df                DataFrame          id                 <...>n\n[379 rows x 9 columns]
dirs              list         n=0
extreme_results   list         n=256
f                 str          california_housing_test.csv
files             list         n=6
os                module       <module 'os' (frozen)>
pd                module       <module 'pandas' from '/u<...>ages/pandas/__init__.py'>
root              str          /content/sample_data
start_idx         int          256


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

print(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


cpu


In [10]:
from transformers import AutoTokenizer
from extended_watermark_processor import WatermarkDetector

detector_tokenizer = AutoTokenizer.from_pretrained("gpt2")

vocab = list(detector_tokenizer.get_vocab().values())

detector = WatermarkDetector(
    vocab=vocab,
    gamma=0.25,
    seeding_scheme="selfhash",
    device=device,
    tokenizer=detector_tokenizer,
    z_threshold=4.0,
    normalizers=[],
    ignore_repeated_ngrams=False,
    select_green_tokens=True
)

print("Detector ready.")

ModuleNotFoundError: No module named 'extended_watermark_processor'

In [11]:
extended_watermark_processor.py

NameError: name 'extended_watermark_processor' is not defined

In [12]:
%cd /content/lm-watermarking

/content/lm-watermarking


In [13]:
import os
print(os.getcwd())
print("extended_watermark_processor.py" in os.listdir())

/content/lm-watermarking
True


In [14]:
from transformers import AutoTokenizer
from extended_watermark_processor import WatermarkDetector

In [15]:
detector_tokenizer = AutoTokenizer.from_pretrained("gpt2")

vocab = list(detector_tokenizer.get_vocab().values())

detector = WatermarkDetector(
    vocab=vocab,
    gamma=0.25,
    seeding_scheme="selfhash",
    device=device,
    tokenizer=detector_tokenizer,
    z_threshold=4.0,
    normalizers=[],
    ignore_repeated_ngrams=False,
    select_green_tokens=True
)

print("Detector ready!")

Detector ready!


In [16]:
sample = df.loc[256, "original_text"]

rewrite = extreme_rewrite_v2(sample)

print(detector.detect(rewrite))

NameError: name 'extreme_rewrite_v2' is not defined

In [17]:
def extreme_rewrite_v2(text):
    prompt = f"""
You are rewriting text to preserve meaning but remove stylistic and wording traces.

Rewrite the text completely.

Rules:
- Do not copy sentence structure.
- Do not copy phrases longer than 3 words.
- Change word choice aggressively.
- Reorder ideas if meaning remains the same.
- Compress repetitive parts.
- Use a different explanatory style.
- Keep factual meaning unchanged.
- Output only the rewritten text.

Text:
{text}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = model.generate(
        **inputs,
        do_sample=True,
        temperature=1.7,
        top_p=0.98,
        top_k=120,
        max_new_tokens=256
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [18]:
sample = df.loc[256, "original_text"]

rewrite = extreme_rewrite_v2(sample)

print(detector.detect(rewrite))

{'num_tokens_scored': 89, 'num_green_tokens': 18, 'green_fraction': 0.20224719101123595, 'z_score': -1.0403831043155778, 'p_value': np.float64(0.8509190259253401), 'z_score_at_T': tensor([ 1.7321,  0.8165,  0.3333,  0.0000, -0.2582, -0.4714, -0.6547, -0.8165,
        -0.1925, -0.3651, -0.5222, -0.6667, -0.8006, -0.9258, -0.4472, -0.5774,
        -0.7001, -0.8165, -0.3974, -0.5164, -0.1260, -0.2462, -0.3612, -0.4714,
        -0.5774, -0.6794, -0.7778, -0.8729, -0.5361, -0.2108, -0.3111, -0.4082,
        -0.1005, -0.1980, -0.2928, -0.3849, -0.4746, -0.1873,  0.0925,  0.0000,
        -0.0902, -0.1782, -0.2641, -0.3482, -0.4303, -0.5108, -0.5895, -0.6667,
        -0.7423, -0.8165, -0.8893, -0.6405, -0.7137, -0.7857, -0.8563, -0.6172,
        -0.6882, -0.4549, -0.5262, -0.5963, -0.6653, -0.4399, -0.5092, -0.5774,
        -0.6445, -0.7107, -0.4937, -0.5601, -0.6255, -0.6901, -0.4796, -0.5443,
        -0.6082, -0.6712, -0.4667, -0.5298, -0.5922, -0.6537, -0.4547, -0.5164,
        -0.5774, -0.

In [19]:
%whos

Variable                Type                          Data/Info
---------------------------------------------------------------
AutoModelForSeq2SeqLM   type                          <class 'transformers.mode<...>o.AutoModelForSeq2SeqLM'>
AutoTokenizer           type                          <class 'transformers.mode<...>tion_auto.AutoTokenizer'>
WatermarkDetector       type                          <class 'extended_watermar<...>essor.WatermarkDetector'>
checkpoint              DataFrame                           id                 <...>\n[256 rows x 10 columns]
detector                WatermarkDetector             <extended_watermark_proce<...>object at 0x78e3fa957fe0>
detector_tokenizer      GPT2Tokenizer                 GPT2Tokenizer(name_or_pat<...>=True, special=True),\n})
device                  str                           cpu
df                      DataFrame                           id                 <...>n\n[379 rows x 9 columns]
dirs                    list                

In [20]:
for i in range(start_idx, len(df)):

    if i % 10 == 0:
        print(f"Running {i+1}/{len(df)}")

    original = df.loc[i, "original_text"]
    before = detector.detect(original)

    try:
        rewritten = extreme_rewrite_v2(original)

        if rewritten is None or len(rewritten.split()) < 20:
            after_z = None
            after_pred = None
            error = "too_short_or_empty"
        else:
            after = detector.detect(rewritten)
            after_z = after["z_score"]
            after_pred = after["prediction"]
            error = None

    except Exception as e:
        rewritten = ""
        after_z = None
        after_pred = None
        error = str(e)

    extreme_results.append({
        "id": df.loc[i, "id"],
        "prompt": df.loc[i, "prompt"],
        "before_z_score": before["z_score"],
        "extreme_z_score": after_z,
        "z_score_drop": before["z_score"] - after_z if after_z is not None else None,
        "before_prediction": before["prediction"],
        "extreme_prediction": after_pred,
        "error": error,
        "original_text": original,
        "extreme_rewrite_text": rewritten
    })

    pd.DataFrame(extreme_results).to_excel(
        "/content/lm-watermarking/experiment3_extreme_rewrite_checkpoint.xlsx",
        index=False
    )

Running 261/379
Running 271/379
Running 281/379
Running 291/379
Running 301/379
Running 311/379
Running 321/379
Running 331/379
Running 341/379
Running 351/379
Running 361/379
Running 371/379


In [21]:
import pandas as pd

extreme_df = pd.DataFrame(extreme_results)

extreme_df.to_excel(
    "/content/experiment3_extreme_rewrite_results.xlsx",
    index=False
)

print("Saved!")

Saved!


In [22]:
from google.colab import files

files.download("/content/experiment3_extreme_rewrite_results.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>